# Phase A.0 - Chargement et Préparation des Données

Ce notebook permet de charger le dataset Yelp et de préparer les données pour les analyses exploratoires.

## Fichiers du dataset:
- `review.json`: Avis textuels associés à une note (1 à 5 étoiles)
- `business.json`: Informations sur les commerces
- `user.json`: Informations sur les utilisateurs

In [13]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configuration de l'affichage
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Affichage complet des DataFrames
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

## 1. Configuration des chemins

In [14]:
# Chemins vers les fichiers de données
DATA_DIR = Path('./data')

REVIEW_FILE = DATA_DIR / 'raw' / 'yelp_academic_reviews4students.jsonl'
BUSINESS_FILE = DATA_DIR / 'raw' / 'yelp_academic_dataset_business.json'
USER_FILE = DATA_DIR / 'raw' / 'yelp_academic_dataset_user4students.jsonl'

# Vérifier que les fichiers existent
for file in [REVIEW_FILE, BUSINESS_FILE, USER_FILE]:
    if file.exists():
        print(f"✓ {file.name} trouvé")
    else:
        print(f"✗ {file.name} MANQUANT")

✓ yelp_academic_reviews4students.jsonl trouvé
✓ yelp_academic_dataset_business.json trouvé
✓ yelp_academic_dataset_user4students.jsonl trouvé


## 2. Fonctions de chargement des données

Les fichiers Yelp sont au format JSON Lines (un objet JSON par ligne).

In [15]:
def load_json_lines(file_path, n_lines=None):
    """
    Charge un fichier JSON Lines dans un DataFrame pandas.
    
    Args:
        file_path: Chemin vers le fichier JSON
        n_lines: Nombre de lignes à charger (None = tout charger)
    
    Returns:
        DataFrame pandas
    """
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if n_lines and i >= n_lines:
                break
            data.append(json.loads(line))
            
            # Afficher la progression toutes les 100k lignes
            if (i + 1) % 100000 == 0:
                print(f"  Chargé {i + 1:,} lignes...")
    
    return pd.DataFrame(data)

## 3. Chargement des données

**Note**: Pour commencer, vous pouvez charger un échantillon des données avec `n_lines`. Une fois que vous avez testé vos analyses, vous pouvez charger l'ensemble complet.

In [16]:
# Paramètres de chargement
SAMPLE_SIZE = None  # Mettre un nombre (ex: 100000) pour un échantillon, None pour tout charger

print("Chargement des reviews...")
df_reviews = load_json_lines(REVIEW_FILE, n_lines=SAMPLE_SIZE)
print(f"✓ {len(df_reviews):,} reviews chargées\n")

print("Chargement des businesses...")
df_business = load_json_lines(BUSINESS_FILE)
print(f"✓ {len(df_business):,} businesses chargés\n")

print("Chargement des users...")
df_users = load_json_lines(USER_FILE)
print(f"✓ {len(df_users):,} users chargés")

Chargement des reviews...
  Chargé 100,000 lignes...
  Chargé 200,000 lignes...
  Chargé 300,000 lignes...
  Chargé 400,000 lignes...
  Chargé 500,000 lignes...
  Chargé 600,000 lignes...
  Chargé 700,000 lignes...
  Chargé 800,000 lignes...
  Chargé 900,000 lignes...
  Chargé 1,000,000 lignes...
✓ 1,000,000 reviews chargées

Chargement des businesses...
  Chargé 100,000 lignes...
✓ 150,346 businesses chargés

Chargement des users...
  Chargé 100,000 lignes...
  Chargé 200,000 lignes...
  Chargé 300,000 lignes...
  Chargé 400,000 lignes...
  Chargé 500,000 lignes...
✓ 558,095 users chargés


## 4. Exploration initiale des données

### 4.1 Reviews

**Le dataset Reviews : yelp_academic_reviews4students**

Le dataset regroupe les avis textuels associés à une note de 1 à 5 : 
- "review_id" : identifiant unique de l'avis
- "user_id" : identifiant de l'utilisateur ayant rédigé l'avis
- "business_id" : identifiant du commerce concerné par l'avis
- "stars" : note attribuée (de 1 à 5 étoiles)
- "useful" : nombre de votes "utile" reçus par l'avis
- "funny" : nombre de votes "drôle" reçus par l'avis
- "cool" : nombre de votes "cool" reçus par l'avis
- "text" : texte de l'avis
- "date" : date de publication de l'avis

In [17]:
print("=== REVIEWS ===")
print(f"Nombre de reviews: {len(df_reviews):,}")
print(f"\nColonnes: {list(df_reviews.columns)}")
print(f"\nTypes de données:")
print(df_reviews.dtypes)
print(f"\nPremières lignes:")
df_reviews.head()

=== REVIEWS ===
Nombre de reviews: 1,000,000

Colonnes: ['review_id', 'user_id', 'business_id', 'stars', 'useful', 'funny', 'cool', 'text', 'date']

Types de données:
review_id      object
user_id        object
business_id    object
stars           int64
useful          int64
funny           int64
cool            int64
text           object
date            int64
dtype: object

Premières lignes:


,review_id,user_id,business_id,stars,useful,funny,cool,text,date
0,J5Q1gH4ACCj6CtQG7Yom7g,56gL9KEJNHiSDUoyjk2o3Q,8yR12PNSMo6FBYx1u5KPlw,2,1,0,0,Went for lunch and found that my burger was me...,1522876193000
1,HlXP79ecTquSVXmjM10QxQ,bAt9OUFX9ZRgGLCXG22UmA,pBNucviUkNsiqhJv5IFpjg,5,0,0,0,I needed a new tires for my wife's car. They h...,1590322934000
2,JBBULrjyGx6vHto2osk_CQ,NRHPcLq2vGWqgqwVugSgnQ,8sf9kv6O4GgEb0j1o22N1g,5,0,0,0,Jim Woltman who works at Goleta Honda is 5 sta...,1550116068000
3,U9-43s8YUl6GWBFCpxUGEw,PAxc0qpqt5c2kA0rjDFFAg,XwepyB7KjJ-XGJf0vKc6Vg,4,0,0,0,Been here a few times to get some shrimp. The...,1367027749000
4,8T8EGa_4Cj12M6w8vRgUsQ,BqPR1Dp5Rb_QYs9_fz9RiA,prm5wvpp0OHJBlrvTj9uOg,5,0,0,0,This is one fantastic place to eat whether you...,1557944965000


In [18]:
# Statistiques descriptives
print("Statistiques sur les reviews:")
df_reviews.describe()

Statistiques sur les reviews:


,stars,useful,funny,cool,date
count,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1.000000e+06
mean,3.749501,1.183582,0.324272,0.497250,1.484213e+12
std,1.478731,3.353949,1.557975,2.223279,9.703951e+10
min,1.000000,0.000000,0.000000,0.000000,1.109902e+12
25%,3.000000,0.000000,0.000000,0.000000,1.422195e+12
50%,4.000000,0.000000,0.000000,0.000000,1.496515e+12
75%,5.000000,1.000000,0.000000,0.000000,1.558629e+12
max,5.000000,997.000000,370.000000,399.000000,1.642622e+12


### 4.2 Business

**Le dataset Business : yelp_academic_dataset_business.json**

Ce dataset regroupe les informations sur les commerces :
- "business_id" : identifiant unique du commerce
- "name" : nom du commerce
- "address" : adresse du commerce
- "city" : ville
- "state" : état/région
- "postal_code" : code postal
- "latitude" : latitude géographique
- "longitude" : longitude géographique
- "stars" : note moyenne du commerce
- "review_count" : nombre d'avis reçus
- "is_open" : indicateur d'ouverture (1 = ouvert, 0 = fermé)
- "attributes" : dictionnaire d'attributs (ex : "WiFi", "Parking", etc.)
- "categories" : catégories du commerce (ex : "Restaurants, Bars")
- "hours" : horaires d'ouverture pour chaque jours de la semaines

In [19]:
print("=== BUSINESS ===")
print(f"Nombre de businesses: {len(df_business):,}")
print(f"\nColonnes: {list(df_business.columns)}")
print(f"\nPremières lignes:")
df_business.head()

=== BUSINESS ===
Nombre de businesses: 150,346

Colonnes: ['business_id', 'name', 'address', 'city', 'state', 'postal_code', 'latitude', 'longitude', 'stars', 'review_count', 'is_open', 'attributes', 'categories', 'hours']

Premières lignes:


,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
0,Pns2l4eNsfO8kk83dixA6A,"Abby Rappoport, LAC, CMQ","1616 Chapala St, Ste 2",Santa Barbara,CA,93101,34.426679,-119.711197,5.0,7,0,{'ByAppointmentOnly': 'True'},"Doctors, Traditional Chinese Medicine, Naturop...",None
1,mpf3x-BjTdTEA3yCZrAYPw,The UPS Store,87 Grasso Plaza Shopping Center,Affton,MO,63123,38.551126,-90.335695,3.0,15,1,{'BusinessAcceptsCreditCards': 'True'},"Shipping Centers, Local Services, Notaries, Ma...","{'Monday': '0:0-0:0', 'Tuesday': '8:0-18:30', ..."
2,tUFrWirKiKi_TAnsVWINQQ,Target,5255 E Broadway Blvd,Tucson,AZ,85711,32.223236,-110.880452,3.5,22,0,"{'BikeParking': 'True', 'BusinessAcceptsCredit...","Department Stores, Shopping, Fashion, Home & G...","{'Monday': '8:0-22:0', 'Tuesday': '8:0-22:0', ..."
3,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,1,"{'RestaurantsDelivery': 'False', 'OutdoorSeati...","Restaurants, Food, Bubble Tea, Coffee & Tea, B...","{'Monday': '7:0-20:0', 'Tuesday': '7:0-20:0', ..."
4,mWMc6_wTdE0EUBKIGXDVfA,Perkiomen Valley Brewery,101 Walnut St,Green Lane,PA,18054,40.338183,-75.471659,4.5,13,1,"{'BusinessAcceptsCreditCards': 'True', 'Wheelc...","Brewpubs, Breweries, Food","{'Wednesday': '14:0-22:0', 'Thursday': '16:0-2..."


In [20]:
# Statistiques descriptives
print("Statistiques sur les Business:")
df_business.describe()

Statistiques sur les Business:


,latitude,longitude,stars,review_count,is_open
count,150346.000000,150346.000000,150346.000000,150346.000000,150346.00000
mean,36.671150,-89.357339,3.596724,44.866561,0.79615
std,5.872759,14.918502,0.974421,121.120136,0.40286
min,27.555127,-120.095137,1.000000,5.000000,0.00000
25%,32.187293,-90.357810,3.000000,8.000000,1.00000
50%,38.777413,-86.121179,3.500000,15.000000,1.00000
75%,39.954036,-75.421542,4.500000,37.000000,1.00000
max,53.679197,-73.200457,5.000000,7568.000000,1.00000


### 4.3 Users

**Le dataset Users : yelp_academic_dataset_user4students.jsonl**

Ce dataset regroupe les informations sur les utilisateurs :
- "user_id" : identifiant unique de l'utilisateur
- "name" : nom de l'utilisateur
- "review_count" : nombre d'avis rédigés
- "yelping_since" : date d'inscription sur Yelp
- "useful" : nombre total de votes "utile" reçus
- "funny" : nombre total de votes "drôle" reçus
- "cool" : nombre total de votes "cool" reçus
- "elite" : années où l'utilisateur a été "élite"
- "friends" : liste des identifiants des amis
- "fans" : nombre de fans
- "average_stars" : note moyenne attribuée par l'utilisateur
- "compliment_hot" : nombre de compliments "hot" reçus
- "compliment_more" : nombre de compliments "more" reçus
- "compliment_profile" : nombre de compliments "profile" reçus
- "compliment_cute" : nombre de compliments "cute" reçus
- "compliment_list" : nombre de compliments "list" reçus
- "compliment_note" : nombre de compliments "note" reçus
- "compliment_plain" : nombre de compliments "plain" reçus
- "compliment_cool" : nombre de compliments "cool" reçus
- "compliment_funny" : nombre de compliments "funny" reçus
- "compliment_writer" : nombre de compliments "writer" reçus
- "compliment_photos" : nombre de compliments "photos" reçus

In [21]:
print("=== USERS ===")
print(f"Nombre d'utilisateurs: {len(df_users):,}")
print(f"\nColonnes: {list(df_users.columns)}")
print(f"\nPremières lignes:")
df_users.head()

=== USERS ===
Nombre d'utilisateurs: 558,095

Colonnes: ['user_id', 'name', 'review_count', 'yelping_since', 'useful', 'funny', 'cool', 'elite', 'friends', 'fans', 'average_stars', 'compliment_hot', 'compliment_more', 'compliment_profile', 'compliment_cute', 'compliment_list', 'compliment_note', 'compliment_plain', 'compliment_cool', 'compliment_funny', 'compliment_writer', 'compliment_photos']

Premières lignes:


,user_id,name,review_count,yelping_since,useful,funny,cool,elite,friends,fans,average_stars,compliment_hot,compliment_more,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos
0,qVc8ODYU5SZjKXVBgXdI7w,Walker,585,2007-01-25 16:47:26,7217,1259,5994,2007,"NSCy54eWehBJyZdG2iE84w, pe42u7DcCH2QmI81NX-8qA...",267,3.91,250,65,55,56,18,232,844,467,467,239,180
1,j14WgRoU_-2ZE1aw1dXrJg,Daniel,4333,2009-01-25 04:35:42,43091,13066,27281,"2009,2010,2011,2012,2013,2014,2015,2016,2017,2...","ueRPE0CX75ePGMqOFVj6IQ, 52oH4DrRvzzl8wh5UXyU0A...",3138,3.74,1145,264,184,157,251,1847,7054,3131,3131,1521,1946
2,SZDeASXq7o05mMNLshsdIA,Gwen,224,2005-11-29 04:38:33,512,330,299,"2009,2010,2011","enx1vVPnfdNUdPho6PH_wg, 4wOcvMLtU6a9Lslggq74Vg...",28,4.27,24,4,1,6,2,12,16,26,26,10,9
3,hA5lMy-EnncsH4JoR-hFGQ,Karen,79,2007-01-05 19:40:59,29,15,7,,"PBK4q9KEEBHhFvSXCUirIw, 3FWPpM7KU1gXeOM_ZbYMbA...",1,3.54,1,1,0,0,0,1,1,0,0,0,0
4,q_QQ5kBBwlCcbL1s4NVK3g,Jane,1221,2005-03-14 20:26:35,14953,9940,11211,"2006,2007,2008,2009,2010,2011,2012,2013,2014","xBDpTUbai0DXrvxCe3X16Q, 7GPNBO496aecrjJfW6UWtg...",1357,3.85,1713,163,191,361,147,1212,5696,2543,2543,815,323


In [22]:
# Statistiques descriptives
print("Statistiques sur les Users:")
df_users.describe()

Statistiques sur les Users:


,review_count,useful,funny,cool,fans,average_stars,compliment_hot,compliment_more,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos
count,558095.000000,558095.000000,558095.000000,558095.000000,558095.000000,558095.000000,558095.000000,558095.000000,558095.000000,558095.000000,558095.000000,558095.000000,558095.000000,558095.000000,558095.000000,558095.000000,558095.000000
mean,40.474331,87.115536,36.315088,53.210582,2.928145,3.688474,3.840511,0.608986,0.405750,0.279884,0.154565,3.160184,6.807312,6.171680,6.171680,2.315832,2.801208
std,129.605063,1083.999707,687.636992,958.281753,31.280800,1.021664,112.316355,23.129695,27.348862,20.576562,18.672886,108.818971,204.316032,153.630569,153.630569,53.039046,166.387901
min,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,4.000000,1.000000,0.000000,0.000000,0.000000,3.180000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,10.000000,6.000000,1.000000,1.000000,0.000000,3.870000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,28.000000,24.000000,5.000000,6.000000,1.000000,4.400000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000
max,17473.000000,206296.000000,185823.000000,199878.000000,12497.000000,5.000000,25784.000000,13501.000000,14180.000000,13654.000000,12669.000000,59031.000000,101097.000000,49967.000000,49967.000000,15934.000000,82630.000000


---

In [23]:
print("=== VALEURS MANQUANTES ===")
print("\nReviews:")
print(df_reviews.isnull().sum())

print("\nBusiness:")
print(df_business.isnull().sum())

print("\nUsers:")
print(df_users.isnull().sum())

=== VALEURS MANQUANTES ===

Reviews:
review_id      0
user_id        0
business_id    0
stars          0
useful         0
funny          0
cool           0
text           0
date           0
dtype: int64

Business:
business_id         0
name                0
address             0
city                0
state               0
postal_code         0
latitude            0
longitude           0
stars               0
review_count        0
is_open             0
attributes      13744
categories        103
hours           23223
dtype: int64

Users:
user_id               0
name                  0
review_count          0
yelping_since         0
useful                0
funny                 0
cool                  0
elite                 0
friends               0
fans                  0
average_stars         0
compliment_hot        0
compliment_more       0
compliment_profile    0
compliment_cute       0
compliment_list       0
compliment_note       0
compliment_plain      0
compliment_cool       0
c